# Atividade: o feed da Copa de 2026

O fornecedor mandou o feed de eventos da **Copa de 2026**, na pasta `data/events_2026`, e as partidas dessa Copa, na
pasta `data/raw_2026`. O time de analytics quer os mesmos números que você entregou nesta aula, agora para 2026.

Faça cada passo sozinho antes de olhar a resposta. **As células de código deste notebook já são o gabarito**: use-as
para conferir o seu caminho.

Três avisos sobre o feed de 2026:

- a seleção vem pela **sigla da FIFA** (`BRA`, `FRA`), e não pelo nome, como no histórico;
- o feed foi **simulado** para o curso a partir das partidas reais de 2026 (o `data/README.md` explica), então não
  compare os passes de 2026 com os do histórico;
- os passes de 2026 ficam numa saída separada, sem juntar com o histórico: juntar exigiria padronizar as siglas antes.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-atividade-2026")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS_2026 = "../../data/events_2026"
RAW_2026 = "../../data/raw_2026"
OUT = "../../data/out/atividade_2026"

spark

## Passo 1. Ler o feed e as partidas de 2026

Leia o feed de 2026 em Parquet, renomeando `MatchID` para `match_id`, e as partidas de 2026 com a `limpar_partidas`
da aula 2. Conte as linhas das duas.

In [ ]:
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

eventos_2026 = spark.read.parquet(EVENTOS_2026).withColumnRenamed("MatchID", "match_id")

partidas_2026 = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW_2026}/WorldCupMatches.csv")
)

print("lances  :", eventos_2026.count())
print("partidas:", partidas_2026.count())

**Resposta:** 312.333 lances e 104 partidas. O feed de 2026 sozinho tem mais lances que o histórico inteiro, porque
traz os passes de todas as partidas.

## Passo 2. Conferir os gols do feed (task 1)

Os gols do feed batem com os placares das partidas de 2026?

In [ ]:
gols_no_feed = eventos_2026.filter(F.col("event_type") == "Goal").count()
gols_nos_placares = partidas_2026.select(F.sum("total_gols")).first()[0]

print("gols no feed     :", gols_no_feed)
print("gols nos placares:", gols_nos_placares)

**Resposta:** 333 nos dois. Os placares de 2026 incluem os gols das disputas de pênaltis, e o feed também (o
`data/README.md` explica).

## Passo 3. O ano em cada lance (task 2)

Junte o ano das partidas aos lances e confira no plano qual join o Spark escolheu.

In [ ]:
lances_2026 = eventos_2026.join(partidas_2026.select("match_id", "ano"), on="match_id")

print("lances com o ano:", lances_2026.count())
lances_2026.explain()

**Resposta:** os mesmos 312.333 lances, nenhum perdido, e um `BroadcastHashJoin`: as 104 partidas são pequenas e vão
para cada executor, como no vídeo 4.2.

## Passo 4. Os passes por partida e por Copa (task 3)

Calcule os passes de cada seleção em cada partida e depois o resumo por Copa, com a média por partida.

In [ ]:
passes_por_partida_2026 = (
    lances_2026
    .filter(F.col("event_type") == "Pass")
    .groupBy("ano", "match_id", "team")
    .agg(F.count("*").alias("passes"))
)

passes_por_copa_2026 = (
    passes_por_partida_2026
    .groupBy("ano", F.col("team").alias("selecao"))
    .agg(
        F.sum("passes").alias("passes"),
        F.count("*").alias("partidas"),
        F.round(F.avg("passes"), 1).alias("media_por_partida"),
    )
)

passes_por_copa_2026.orderBy(F.desc("media_por_partida")).show(5)

**Resposta:** 48 seleções. Sem o filtro de ano do vídeo 4.3: aqui só existe uma Copa.

## Passo 5. Gravar as duas granularidades em Parquet (task 4)

In [ ]:
passes_por_partida_2026.write.mode("overwrite").parquet(f"{OUT}/passes_por_partida")
passes_por_copa_2026.write.mode("overwrite").parquet(f"{OUT}/passes_por_copa")

print("por partida:", spark.read.parquet(f"{OUT}/passes_por_partida").count(), "linhas")
print("por Copa   :", spark.read.parquet(f"{OUT}/passes_por_copa").count(), "linhas")

**Resposta:** 208 linhas por partida (104 partidas, 2 seleções) e 48 por Copa.

## Desafio: quantas partições?

Compare o número de partições da leitura do feed histórico com o da leitura do feed de 2026. Por que são diferentes?
Liste as duas pastas para responder.

In [ ]:
historico = spark.read.parquet("../../data/events")

print("partições do histórico:", historico.rdd.getNumPartitions())
print("partições de 2026     :", eventos_2026.rdd.getNumPartitions())

!ls ../../data/events ../../data/events_2026

**Resposta:** o histórico é um arquivo só, pequeno, e vira uma partição. O feed de 2026 chegou em 8 arquivos, e o Spark
lê vários arquivos em paralelo, um por núcleo, até o número de arquivos. Com 8 núcleos ou mais, são 8 partições; com
menos, o número de núcleos. É a equipe da aula 1: com mais pilhas, mais ajudantes trabalham ao mesmo tempo.

## Encerrando

In [ ]:
spark.stop()

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()